# House Prices: TotalSF 단일 feature 실험

## tl;dr

- Raw-year reference에 TotalSF 하나만 추가했다.
- Standalone CV RMSLE는 0.158196 ± 0.044130다.
- Raw reference 대비 CV +0.007824, OOF +0.008335로 악화됐다.
- 개선 fold는 0/5이므로 standalone feature는 reject다.
- OOF 잔차 보완성은 별도 raw+feature blend 후보에서 평가한다.

## Context & Methods

원본 YearRemodAdd를 그대로 사용하는 raw-year reference를 고정하고,
지하실·1층·2층 면적 합계인 TotalSF 하나만 추가한다.

### Key Assumptions

- 생성에는 Id와 SalePrice를 사용하지 않는다.
- 원본 feature는 수정하지 않고 새 수치 feature 하나만 추가한다.
- 모든 행을 유지하며 전처리는 fold 학습 부분에서만 적합한다.
- fold, seed, 모델, optimizer, loss, batch size, early stopping은 reference와 같다.
- raw reference보다 입력 차원이 하나 늘어 초기화 경로가 달라지는 caveat가 있다.
  세 후보끼리는 추가 차원 수와 위치가 같아 상호 비교가 더 직접적이다.

## Data

data/train.csv, data/test.csv, data/sample_submission.csv를 사용한다.
구조적 부재 문자열은 keep_default_na=False로 보존한다.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent.parent

TRAIN_PATH = ROOT / "data" / "train.csv"
EXPERIMENTS_PATH = ROOT / "reports" / "experiments.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "data" / "sample_submission.csv"
NOTEBOOK_PATH = ROOT / "notebooks/feature_engineering/basemlp_total_sf.ipynb"
SUMMARY_PATH = ROOT / "reports/basemlp_total_sf_summary.json"
COMPARISON_PATH = ROOT / "reports/basemlp_total_sf_comparison.json"
RAW_REFERENCE_EXPERIMENT_ID = "BASEMLP-20260719-RAWYEAR-ONLY-NB-04"
RAW_REFERENCE_METRICS_PATH = (
    ROOT / "reports" / f"basemlp_metrics_{RAW_REFERENCE_EXPERIMENT_ID}.json"
)
RAW_REFERENCE_SUBMISSION_PATH = (
    ROOT / "submissions" /
    f"submission_{RAW_REFERENCE_EXPERIMENT_ID}-SUB-01.csv"
)
BASELINE_SUBMISSION_PATH = (
    ROOT / "submissions" / "submission_BASEMLP-20260719-2H-01.csv"
)
EXPERIMENT_ID = "BASEMLP-20260719-FEAT-TOTALSF-NB-06"
SUBMISSION_ID = f"{EXPERIMENT_ID}-SUB-01"
SUBMISSION_PATH = ROOT / "submissions" / f"submission_{SUBMISSION_ID}.csv"
FEATURE_NAME = "TotalSF"
FEATURE_DESCRIPTION = "지하실·1층·2층 면적 합계"
FEATURE_INPUTS = ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']

SEED = 42
N_SPLITS = 5
HIDDEN_DIMS = (128, 64)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 64
MAX_EPOCHS = 200
PATIENCE = 25
MIN_DELTA = 1e-5
DEVICE = torch.device("cpu")

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]
BASEMENT_COLUMNS = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
class BaseMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden1.bias)
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden2.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def predict_log_target(
    model: BaseMLP,
    features: np.ndarray,
    target_mean: float,
    target_std: float,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
        standardized = model(tensor).cpu().numpy().astype(np.float64)
    return standardized * target_std + target_mean


def train_fold(
    X_train: np.ndarray,
    y_train_log: np.ndarray,
    X_valid: np.ndarray,
    y_valid_log: np.ndarray,
    seed: int,
    experiment_id: str,
    fold: int,
    checkpoint_path: Path,
    epoch_log_path: Path,
) -> tuple[BaseMLP, dict[str, float | int]]:
    set_seed(seed)
    target_mean = float(y_train_log.mean())
    target_std = float(y_train_log.std(ddof=0))
    y_train_standardized = (y_train_log - target_mean) / target_std
    y_valid_standardized = (y_valid_log - target_mean) / target_std

    train_dataset = TensorDataset(
        torch.as_tensor(X_train, dtype=torch.float32),
        torch.as_tensor(y_train_standardized, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    X_valid_tensor = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    y_valid_tensor = torch.as_tensor(
        y_valid_standardized, dtype=torch.float32, device=DEVICE
    )

    model = BaseMLP(X_train.shape[1]).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_score = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    epoch_rows: list[dict[str, float | int]] = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for batch_features, batch_target in train_loader:
            batch_features = batch_features.to(DEVICE)
            batch_target = batch_target.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(batch_features)
            loss = loss_fn(prediction, batch_target)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach().cpu()) * len(batch_features)
            train_count += len(batch_features)

        model.eval()
        with torch.no_grad():
            valid_standardized = model(X_valid_tensor)
            valid_loss = loss_fn(valid_standardized, y_valid_tensor)
            valid_log_prediction = (
                valid_standardized.cpu().numpy().astype(np.float64) * target_std
                + target_mean
            )
        valid_rmsle = float(
            np.sqrt(np.mean((y_valid_log - valid_log_prediction) ** 2))
        )
        epoch_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_count,
                "validation_loss": float(valid_loss.cpu()),
                "validation_rmsle": valid_rmsle,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
            }
        )

        if valid_rmsle < best_score - MIN_DELTA:
            best_score = valid_rmsle
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "experiment_id": experiment_id,
                    "fold": fold,
                    "input_dim": int(X_train.shape[1]),
                    "model_state_dict": model.state_dict(),
                    "target_mean": target_mean,
                    "target_std": target_std,
                    "architecture": [*HIDDEN_DIMS, 1],
                    "seed": seed,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            break

    pd.DataFrame(epoch_rows).to_csv(epoch_log_path, index=False)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    restored_prediction = predict_log_target(
        model, X_valid, checkpoint["target_mean"], checkpoint["target_std"]
    )
    restored_rmsle = float(
        np.sqrt(np.mean((y_valid_log - restored_prediction) ** 2))
    )
    if not np.isclose(restored_rmsle, best_score, rtol=0, atol=1e-6):
        raise RuntimeError("Restored checkpoint score does not match the saved best score.")
    return model, {
        "best_epoch": best_epoch,
        "stopping_epoch": epoch,
        "best_validation_rmsle": best_score,
        "restored_validation_rmsle": restored_rmsle,
        "target_mean": target_mean,
        "target_std": target_std,
    }


def append_experiment(row: dict[str, str]) -> None:
    with EXPERIMENTS_PATH.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        existing_ids = {existing["experiment_id"] for existing in reader}
    if fieldnames is None:
        raise RuntimeError("reports/experiments.csv has no header.")
    if row["experiment_id"] in existing_ids:
        raise RuntimeError(f"Experiment ID already exists: {row['experiment_id']}")
    with EXPERIMENTS_PATH.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writerow({field: row.get(field, "") for field in fieldnames})


train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
assert train.shape == (1460, 81)
assert train["Id"].is_unique
assert train["SalePrice"].gt(0).all()
categorical_columns = [
    column
    for column in train.columns
    if column not in {*NUMERIC_COLUMNS, "Id", "SalePrice"}
]
assert len(NUMERIC_COLUMNS) == 34
assert len(categorical_columns) == 45

y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)
print(f"train: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"train sha256: {sha256(TRAIN_PATH)}")
print("external project script imports: 0")

train: 1,460 rows × 81 columns
train sha256: 1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b063e8f692a5f929d41
external project script imports: 0


### Single-feature audit

원본 열을 바꾸지 않고 지정한 feature 하나만 추가됐는지 확인한다.

In [2]:
def clean_rows_without_record_correction(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )

    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"

    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace({"NA": label, "": label})

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(
        lambda value: f"SC{value}"
    )
    cleaned["MoSold"] = cleaned["MoSold"].map(
        lambda value: f"M{int(value):02d}"
    )
    for column in categorical_columns:
        cleaned[column] = cleaned[column].replace(
            {"NA": "Unknown", "": "Unknown"}
        )
    return cleaned

def add_single_feature(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    featured = clean_rows_without_record_correction(
        frame, categorical_columns
    )
    components = ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]
    featured[FEATURE_NAME] = (
        featured[components].fillna(0).sum(axis=1)
    )
    return featured


raw_year_X = clean_rows_without_record_correction(
    train.drop(columns="SalePrice"), categorical_columns
)
feature_X = add_single_feature(
    train.drop(columns="SalePrice"), categorical_columns
)

assert FEATURE_NAME not in raw_year_X.columns
assert FEATURE_NAME in feature_X.columns
assert raw_year_X.equals(feature_X.drop(columns=FEATURE_NAME))
assert not (set(FEATURE_INPUTS) & {"Id", "SalePrice"})
assert feature_X[FEATURE_NAME].notna().all()
assert np.isfinite(feature_X[FEATURE_NAME]).all()
assert feature_X[FEATURE_NAME].ge(0).all()
assert feature_X[FEATURE_NAME].nunique() > 1

display(pd.Series({
    "feature": FEATURE_NAME,
    "input columns": ", ".join(FEATURE_INPUTS),
    "generated features": 1,
    "original columns modified": 0,
    "row-specific rules": 0,
    "unique feature values": int(
        feature_X[FEATURE_NAME].nunique()
    ),
}, name="single-feature audit"))


feature                                              TotalSF
input columns                TotalBsmtSF, 1stFlrSF, 2ndFlrSF
generated features                                         1
original columns modified                                  0
row-specific rules                                         0
unique feature values                                    963
Name: single-feature audit, dtype: object

## Results

검증된 raw-year reference와 동일한 학습·검증 조건으로 실행한다.

In [3]:
def make_preprocessor(numeric_columns: list[str]) -> ColumnTransformer:
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="constant", fill_value="Unknown"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, numeric_columns),
            ("categorical", categorical, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


def run_experiment(
    experiment_id: str,
    X: pd.DataFrame,
    numeric_columns: list[str],
) -> dict[str, object]:
    started = time.perf_counter()
    checkpoint_dir = ROOT / "artifacts" / "checkpoints" / experiment_id
    preprocessor_dir = ROOT / "artifacts" / "preprocessors" / experiment_id
    epoch_log_dir = ROOT / "artifacts" / "logs" / experiment_id
    for directory in (checkpoint_dir, preprocessor_dir, epoch_log_dir):
        directory.mkdir(parents=True, exist_ok=False)

    metrics_path = ROOT / "reports" / f"basemlp_metrics_{experiment_id}.json"
    oof_path = ROOT / "reports" / f"basemlp_oof_{experiment_id}.csv"
    folds = list(
        KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=SEED,
        ).split(X)
    )
    oof_log_prediction = np.full(len(X), np.nan, dtype=np.float64)
    fold_rows: list[dict[str, object]] = []

    for fold, (train_index, valid_index) in enumerate(folds, start=1):
        fold_seed = SEED + fold
        checkpoint_path = checkpoint_dir / f"fold_{fold}_best.pt"
        preprocessor_path = (
            preprocessor_dir / f"fold_{fold}_preprocessor.joblib"
        )
        epoch_log_path = epoch_log_dir / f"fold_{fold}_epochs.csv"

        preprocessor = make_preprocessor(numeric_columns)
        X_train = preprocessor.fit_transform(
            X.iloc[train_index]
        ).astype(np.float32)
        X_valid = preprocessor.transform(
            X.iloc[valid_index]
        ).astype(np.float32)
        joblib.dump(preprocessor, preprocessor_path)

        _, training = train_fold(
            X_train,
            y_log[train_index],
            X_valid,
            y_log[valid_index],
            seed=fold_seed,
            experiment_id=experiment_id,
            fold=fold,
            checkpoint_path=checkpoint_path,
            epoch_log_path=epoch_log_path,
        )

        saved_preprocessor = joblib.load(preprocessor_path)
        X_valid_saved = saved_preprocessor.transform(
            X.iloc[valid_index]
        ).astype(np.float32)
        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )
        restored_model = BaseMLP(checkpoint["input_dim"]).to(DEVICE)
        restored_model.load_state_dict(checkpoint["model_state_dict"])
        valid_prediction = predict_log_target(
            restored_model,
            X_valid_saved,
            checkpoint["target_mean"],
            checkpoint["target_std"],
        )
        oof_log_prediction[valid_index] = valid_prediction
        fold_rmsle = float(
            np.sqrt(
                np.mean(
                    (y_log[valid_index] - valid_prediction) ** 2
                )
            )
        )
        fold_rows.append(
            {
                "fold": fold,
                "train_rows": int(len(train_index)),
                "validation_rows": int(len(valid_index)),
                "encoded_features": int(X_train.shape[1]),
                "seed": fold_seed,
                "best_epoch": int(training["best_epoch"]),
                "stopping_epoch": int(training["stopping_epoch"]),
                "rmsle": fold_rmsle,
            }
        )

    assert not np.isnan(oof_log_prediction).any()
    fold_scores = np.array(
        [row["rmsle"] for row in fold_rows], dtype=np.float64
    )
    cv_mean = float(fold_scores.mean())
    cv_std = float(fold_scores.std(ddof=1))
    oof_rmsle = float(
        np.sqrt(np.mean((y_log - oof_log_prediction) ** 2))
    )
    duration_seconds = time.perf_counter() - started

    pd.DataFrame(
        {
            "Id": train["Id"],
            "SalePrice": y,
            "actual_log1p": y_log,
            "oof_log_prediction": oof_log_prediction,
            "oof_saleprice_prediction": np.clip(
                np.expm1(oof_log_prediction), 0, None
            ),
        }
    ).to_csv(oof_path, index=False)

    metrics = {
        "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_id": experiment_id,
        "source": {
            "train_path": "data/train.csv",
            "train_sha256": sha256(TRAIN_PATH),
            "rows": int(len(train)),
        },
        "features": {
            "generated": [FEATURE_NAME],
            "feature_description": FEATURE_DESCRIPTION,
            "feature_inputs": FEATURE_INPUTS,
            "row_specific_corrections": 0,
            "original_columns_modified": 0,
        },
        "validation": {
            "splitter": "KFold",
            "n_splits": N_SPLITS,
            "shuffle": True,
            "random_state": SEED,
            "metric": "RMSLE / RMSE on log1p target",
            "all_rows_retained": True,
        },
        "model": {
            "class": "BaseMLP(nn.Module)",
            "architecture": f"input->{list(HIDDEN_DIMS)}->1",
            "optimizer": "Adam",
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "loss": "MSELoss on fold-standardized log1p(SalePrice)",
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "min_delta": MIN_DELTA,
            "device": str(DEVICE),
            "precision": "float32",
        },
        "results": {
            "fold_scores": fold_scores.tolist(),
            "cv_mean": cv_mean,
            "cv_std": cv_std,
            "oof_rmsle": oof_rmsle,
            "duration_seconds": duration_seconds,
        },
        "folds": fold_rows,
        "artifacts": {
            "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
            "metrics": str(metrics_path.relative_to(ROOT)),
            "oof_predictions": str(oof_path.relative_to(ROOT)),
            "checkpoint_dir": str(checkpoint_dir.relative_to(ROOT)),
            "preprocessor_dir": str(preprocessor_dir.relative_to(ROOT)),
            "epoch_log_dir": str(epoch_log_dir.relative_to(ROOT)),
        },
        "kaggle_scores": {"public": "unverified", "private": "unverified"},
    }
    metrics_path.write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return metrics

experiment_metrics = run_experiment(
    EXPERIMENT_ID,
    feature_X,
    [*NUMERIC_COLUMNS, FEATURE_NAME],
)


In [4]:
results = experiment_metrics["results"]
raw_reference_metrics = json.loads(
    RAW_REFERENCE_METRICS_PATH.read_text(encoding="utf-8")
)
raw_results = raw_reference_metrics["results"]
cv_delta = float(results["cv_mean"] - raw_results["cv_mean"])
oof_delta = float(
    results["oof_rmsle"] - raw_results["oof_rmsle"]
)
fold_deltas = (
    np.asarray(results["fold_scores"], dtype=np.float64)
    - np.asarray(raw_results["fold_scores"], dtype=np.float64)
)
improved_folds = int((fold_deltas < 0).sum())
decision = "provisional_keep" if cv_delta < 0 else "reject"

comparison_table = pd.DataFrame([
    {
        "experiment_id": RAW_REFERENCE_EXPERIMENT_ID,
        "condition": "raw-year reference",
        "cv_mean": raw_results["cv_mean"],
        "cv_std": raw_results["cv_std"],
        "oof_rmsle": raw_results["oof_rmsle"],
    },
    {
        "experiment_id": EXPERIMENT_ID,
        "condition": FEATURE_NAME,
        "cv_mean": results["cv_mean"],
        "cv_std": results["cv_std"],
        "oof_rmsle": results["oof_rmsle"],
    },
])
fold_table = pd.DataFrame({
    "fold": range(1, N_SPLITS + 1),
    "raw_reference_rmsle": raw_results["fold_scores"],
    "feature_rmsle": results["fold_scores"],
    "feature_minus_reference": fold_deltas,
})
display(comparison_table.round(6))
display(fold_table.round(6))

run_timestamp = datetime.now(timezone.utc).isoformat()
append_experiment({
    "experiment_id": EXPERIMENT_ID,
    "datetime": run_timestamp,
    "objective": "Evaluate TotalSF: 지하실·1층·2층 면적 합계",
    "data_version": f"train sha256={sha256(TRAIN_PATH)}",
    "split_strategy": "KFold(n_splits=5,shuffle=True,random_state=42)",
    "folds": str(N_SPLITS),
    "seed": str(SEED),
    "metric": "RMSLE / RMSE on log1p target",
    "preprocessing": "Raw YearRemodAdd; no row-specific correction; row-only feature; fold median/scaling/one-hot; fold-train target standardization",
    "features": f"79 original predictors + {FEATURE_NAME}; Id excluded",
    "model": "BaseMLP",
    "architecture": "input->[128,64]->1; ReLU; no dropout; no batch normalization",
    "optimizer": "Adam",
    "loss": "MSELoss on fold-standardized log1p(SalePrice)",
    "learning_rate": str(LEARNING_RATE),
    "batch_size": str(BATCH_SIZE),
    "max_epochs": str(MAX_EPOCHS),
    "patience": str(PATIENCE),
    "best_epoch": "folds=" + "|".join(
        str(row["best_epoch"]) for row in experiment_metrics["folds"]
    ),
    "hyperparameters": json.dumps({
        "reference_experiment": RAW_REFERENCE_EXPERIMENT_ID,
        "generated_features": [FEATURE_NAME],
        "feature_inputs": FEATURE_INPUTS,
        "hidden_dims": list(HIDDEN_DIMS),
        "activation": "ReLU",
        "dropout": 0.0,
        "batch_normalization": False,
        "weight_decay": WEIGHT_DECAY,
        "target_standardization": "fold-train",
        "fold_seeds": list(range(SEED + 1, SEED + N_SPLITS + 1)),
        "all_rows_retained": True,
        "source_notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
        "external_script_dependency": False,
        "added_dimension_caveat": True,
    }, ensure_ascii=False),
    "cv_mean": str(results["cv_mean"]),
    "cv_std": str(results["cv_std"]),
    "checkpoint_path": experiment_metrics["artifacts"]["checkpoint_dir"],
    "artifact_path": "; ".join([
        str(NOTEBOOK_PATH.relative_to(ROOT)),
        experiment_metrics["artifacts"]["metrics"],
        experiment_metrics["artifacts"]["oof_predictions"],
    ]),
    "result": decision,
    "interpretation": (
        f"Raw reference delta: CV={cv_delta:+.6f}; "
        f"OOF={oof_delta:+.6f}; improved folds={improved_folds}/5."
    ),
    "next_action": "Compare all three one-feature candidates before choosing submission order.",
})

summary = {
    "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
    "source": {
        "train_path": "data/train.csv",
        "train_sha256": sha256(TRAIN_PATH),
        "train_rows": int(len(train)),
    },
    "reference_experiment": RAW_REFERENCE_EXPERIMENT_ID,
    "experiment": EXPERIMENT_ID,
    "feature": {
        "name": FEATURE_NAME,
        "description": FEATURE_DESCRIPTION,
        "inputs": FEATURE_INPUTS,
        "row_specific_corrections": 0,
        "other_generated_features": [],
    },
    "raw_reference_results": raw_results,
    "feature_results": results,
    "comparison": {
        "cv_delta": cv_delta,
        "oof_delta": oof_delta,
        "fold_deltas": fold_deltas.tolist(),
        "improved_folds": improved_folds,
        "decision": decision,
        "added_dimension_caveat": True,
    },
    "validation": {
        "notebook_executed_top_to_bottom": True,
        "external_script_dependency": False,
        "kaggle_scores": "unverified",
    },
}
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(
    f"feature={FEATURE_NAME}; decision={decision}; "
    f"CV delta={cv_delta:+.6f}; OOF delta={oof_delta:+.6f}"
)


,experiment_id,condition,cv_mean,cv_std,oof_rmsle
0,BASEMLP-20260719-RAWYEAR-ONLY-NB-04,raw-year reference,0.150372,0.040674,0.154710
1,BASEMLP-20260719-FEAT-TOTALSF-NB-06,TotalSF,0.158196,0.044130,0.163046


,fold,raw_reference_rmsle,feature_rmsle,feature_minus_reference
0,1,0.141188,0.149326,0.008138
1,2,0.131368,0.142144,0.010776
2,3,0.222249,0.235381,0.013132
3,4,0.133601,0.140130,0.006529
4,5,0.123455,0.123998,0.000543


feature=TotalSF; decision=reject; CV delta=+0.007824; OOF delta=+0.008335


## Submission comparison

복원한 5개 checkpoint로 submission을 만들고 raw-year reference와 비교한다.

In [5]:
test = pd.read_csv(TEST_PATH, keep_default_na=False)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
raw_reference_submission = pd.read_csv(
    RAW_REFERENCE_SUBMISSION_PATH
)
baseline_submission = pd.read_csv(BASELINE_SUBMISSION_PATH)

assert test.shape == (1459, 80)
assert list(test.columns) == [
    column for column in train.columns if column != "SalePrice"
]
assert list(sample_submission.columns) == ["Id", "SalePrice"]
assert sample_submission["Id"].equals(test["Id"])
assert raw_reference_submission["Id"].equals(test["Id"])
assert baseline_submission["Id"].equals(test["Id"])

feature_test = add_single_feature(test, categorical_columns)
assert FEATURE_NAME in feature_test.columns
assert np.isfinite(feature_test[FEATURE_NAME]).all()
assert feature_test[FEATURE_NAME].ge(0).all()

checkpoint_dir = ROOT / experiment_metrics["artifacts"]["checkpoint_dir"]
preprocessor_dir = ROOT / experiment_metrics["artifacts"]["preprocessor_dir"]
fold_test_log_predictions = []

for fold in range(1, N_SPLITS + 1):
    preprocessor = joblib.load(
        preprocessor_dir / f"fold_{fold}_preprocessor.joblib"
    )
    checkpoint = torch.load(
        checkpoint_dir / f"fold_{fold}_best.pt",
        map_location=DEVICE,
        weights_only=False,
    )
    X_test = preprocessor.transform(feature_test).astype(np.float32)
    model = BaseMLP(checkpoint["input_dim"]).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    fold_test_log_predictions.append(
        predict_log_target(
            model,
            X_test,
            checkpoint["target_mean"],
            checkpoint["target_std"],
        )
    )

mean_test_log_prediction = np.vstack(
    fold_test_log_predictions
).mean(axis=0)
feature_submission = sample_submission.copy()
feature_submission["SalePrice"] = np.clip(
    np.expm1(mean_test_log_prediction), 0, None
)
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
feature_submission.to_csv(SUBMISSION_PATH, index=False)

checks = {
    "columns": feature_submission.columns.tolist() == ["Id", "SalePrice"],
    "rows": len(feature_submission) == len(test),
    "id_order": feature_submission["Id"].equals(test["Id"]),
    "unique_ids": bool(feature_submission["Id"].is_unique),
    "finite_predictions": bool(
        np.isfinite(feature_submission["SalePrice"]).all()
    ),
    "positive_predictions": bool(
        feature_submission["SalePrice"].gt(0).all()
    ),
}
assert all(checks.values())

difference_vs_raw = (
    feature_submission["SalePrice"].to_numpy(dtype=np.float64)
    - raw_reference_submission["SalePrice"].to_numpy(dtype=np.float64)
)
difference_vs_baseline = (
    feature_submission["SalePrice"].to_numpy(dtype=np.float64)
    - baseline_submission["SalePrice"].to_numpy(dtype=np.float64)
)
absolute_vs_raw = np.abs(difference_vs_raw)
submission_comparison = {
    "submission_experiment_id": SUBMISSION_ID,
    "source_experiment_id": EXPERIMENT_ID,
    "reference_experiment_id": RAW_REFERENCE_EXPERIMENT_ID,
    "feature_name": FEATURE_NAME,
    "submission_path": str(SUBMISSION_PATH.relative_to(ROOT)),
    "raw_reference_submission_path": str(
        RAW_REFERENCE_SUBMISSION_PATH.relative_to(ROOT)
    ),
    "baseline_submission_path": str(
        BASELINE_SUBMISSION_PATH.relative_to(ROOT)
    ),
    "columns_equal_to_reference": (
        feature_submission.columns.tolist()
        == raw_reference_submission.columns.tolist()
    ),
    "id_exact_equal_to_reference": feature_submission["Id"].equals(
        raw_reference_submission["Id"]
    ),
    "different_rows_vs_raw_reference": int(
        np.count_nonzero(difference_vs_raw)
    ),
    "mean_abs_difference_vs_raw_reference": float(
        absolute_vs_raw.mean()
    ),
    "median_abs_difference_vs_raw_reference": float(
        np.median(absolute_vs_raw)
    ),
    "max_abs_difference_vs_raw_reference": float(
        absolute_vs_raw.max()
    ),
    "different_rows_vs_original_baseline": int(
        np.count_nonzero(difference_vs_baseline)
    ),
    "submission_checks": checks,
    "kaggle_public_score": "unverified",
    "kaggle_private_score": "unverified",
}
COMPARISON_PATH.write_text(
    json.dumps(
        submission_comparison, ensure_ascii=False, indent=2
    ),
    encoding="utf-8",
)

append_experiment({
    "experiment_id": SUBMISSION_ID,
    "datetime": datetime.now(timezone.utc).isoformat(),
    "objective": "Create the TotalSF BaseMLP submission candidate",
    "data_version": (
        f"train sha256={sha256(TRAIN_PATH)}; "
        f"test sha256={sha256(TEST_PATH)}"
    ),
    "split_strategy": "KFold(n_splits=5,shuffle=True,random_state=42)",
    "folds": str(N_SPLITS),
    "seed": str(SEED),
    "metric": "RMSLE / RMSE on log1p target",
    "preprocessing": "Raw YearRemodAdd; no row-specific correction; one generated feature; fold median/scaling/one-hot; fold-train target standardization",
    "features": f"79 original predictors + {FEATURE_NAME}; Id excluded",
    "model": "BaseMLP",
    "architecture": "input->[128,64]->1; ReLU; no dropout; no batch normalization",
    "optimizer": "Adam",
    "loss": "MSELoss on fold-standardized log1p(SalePrice)",
    "learning_rate": str(LEARNING_RATE),
    "batch_size": str(BATCH_SIZE),
    "max_epochs": str(MAX_EPOCHS),
    "patience": str(PATIENCE),
    "best_epoch": "folds=" + "|".join(
        str(row["best_epoch"]) for row in experiment_metrics["folds"]
    ),
    "hyperparameters": json.dumps({
        "source_experiment": EXPERIMENT_ID,
        "reference_experiment": RAW_REFERENCE_EXPERIMENT_ID,
        "feature": FEATURE_NAME,
        "test_prediction_strategy": (
            "mean of five restored-checkpoint log predictions then expm1"
        ),
        "external_script_dependency": False,
    }, ensure_ascii=False),
    "cv_mean": str(results["cv_mean"]),
    "cv_std": str(results["cv_std"]),
    "checkpoint_path": experiment_metrics["artifacts"]["checkpoint_dir"],
    "artifact_path": "; ".join([
        str(NOTEBOOK_PATH.relative_to(ROOT)),
        str(SUBMISSION_PATH.relative_to(ROOT)),
        str(COMPARISON_PATH.relative_to(ROOT)),
        experiment_metrics["artifacts"]["metrics"],
        experiment_metrics["artifacts"]["oof_predictions"],
    ]),
    "result": (
        "submission_candidate"
        if decision == "provisional_keep"
        else "submission_candidate_local_worse"
    ),
    "interpretation": (
        f"Raw reference delta: CV={cv_delta:+.6f}; "
        f"different test rows="
        f"{submission_comparison['different_rows_vs_raw_reference']}."
    ),
    "next_action": "Use only after reviewing all three candidates under the same setup.",
})

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
summary["submission"] = submission_comparison
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
display(pd.Series(
    submission_comparison, name="submission comparison"
))


submission_experiment_id                         BASEMLP-20260719-FEAT-TOTALSF-NB-06-SUB-01
source_experiment_id                                    BASEMLP-20260719-FEAT-TOTALSF-NB-06
reference_experiment_id                                 BASEMLP-20260719-RAWYEAR-ONLY-NB-04
feature_name                                                                        TotalSF
submission_path                           submissions/submission_BASEMLP-20260719-FEAT-T...
raw_reference_submission_path             submissions/submission_BASEMLP-20260719-RAWYEA...
baseline_submission_path                  submissions/submission_BASEMLP-20260719-2H-01.csv
columns_equal_to_reference                                                             True
id_exact_equal_to_reference                                                            True
different_rows_vs_raw_reference                                                        1459
mean_abs_difference_vs_raw_reference                                            

## Takeaways

- 생성 feature는 TotalSF 하나이며 원본 열과 행은 수정하지 않았다.
- Standalone CV 변화는 +0.007824, OOF 변화는 +0.008335다.
- 개선 fold는 0/5으로 standalone 모델은 reject다.
- 이 모델의 OOF와 test 예측은 별도 blend 후보의 보조 예측으로만 사용한다.
- 외부 프로젝트 스크립트 의존은 0건이고 Kaggle 점수는 unverified다.